# Renewable Energy Sources in India — Final Working Model

**One-cell implementation:** upload the CSV, preprocess it, train and evaluate the Gradient Boosting model, then enter a sample or custom input to predict next-year total renewable capacity.

The capacity unit is not specified in the supplied CSV, so the notebook does not assume MW or GW. The sample uses Karnataka's 2024 values to demonstrate a 2025 estimate. Population and GSDP are sample inputs, not verified 2025 observations.

In [ ]:
# ================================================================
# RENEWABLE ENERGY SOURCES IN INDIA — FINAL WORKING ML MODEL
# One-cell Google Colab implementation
# Upload the original CSV, train the model, evaluate it, and predict
# next-year total renewable capacity from a sample/custom input.
# ================================================================

!pip -q install pandas numpy scikit-learn joblib

import io, json, warnings
import numpy as np
import pandas as pd
import joblib
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# 1. UPLOAD DATASET
print("Upload renewable_energy_dataset.csv")
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise ValueError("No CSV file uploaded. Please upload renewable_energy_dataset.csv")
df = pd.read_csv(io.BytesIO(uploaded[csv_names[0]]))
print("\nDataset loaded:", df.shape)

# 2. DEFINE COLUMNS AND VALIDATE DATA
CAPACITY_COLUMNS = [
    "Installed - Bio-Mass Power",
    "Installed - Small Hydro Power",
    "Installed - Solar Power",
    "Installed - Wind Power",
    "Installed - Waste to Energy",
]
TARGET = "Total_Renewable_Capacity"
PREVIOUS_TOTAL = "Previous Total Capacity"
LAG_COLUMNS = [f"Previous Year - {c}" for c in CAPACITY_COLUMNS]
REQUIRED = ["State", "Year", "Population", "GSDP (crore INR)", *CAPACITY_COLUMNS]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing columns: {missing}")

for col in ["Year", "Population", "GSDP (crore INR)", *CAPACITY_COLUMNS]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["State", "Year", *CAPACITY_COLUMNS]).copy()
df = df.drop_duplicates(subset=["State", "Year"], keep="last")
if (df[CAPACITY_COLUMNS] < 0).any().any():
    raise ValueError("Negative installed-capacity values were found. Check the source data.")
df[TARGET] = df[CAPACITY_COLUMNS].sum(axis=1)

print("Rows after basic cleaning:", len(df))
print("States/regions:", df["State"].nunique())
print("Years:", int(df["Year"].min()), "to", int(df["Year"].max()))
print("Missing values by column:")
print(df[REQUIRED].isna().sum().to_string())

# 3. CREATE PREVIOUS-YEAR FEATURES (NO CURRENT-YEAR CAPACITY LEAKAGE)
data = df.sort_values(["State", "Year"]).copy()
for source_col, lag_col in zip(CAPACITY_COLUMNS, LAG_COLUMNS):
    data[lag_col] = data.groupby("State")[source_col].shift(1)
data[PREVIOUS_TOTAL] = data.groupby("State")[TARGET].shift(1)
data["Capacity Change"] = data[TARGET] - data[PREVIOUS_TOTAL]
data = data.dropna(subset=[*LAG_COLUMNS, PREVIOUS_TOTAL]).copy()

FEATURES = [
    "State", "Year", "Population", "GSDP (crore INR)",
    PREVIOUS_TOTAL, *LAG_COLUMNS
]
NUMERIC_FEATURES = [
    "Year", "Population", "GSDP (crore INR)",
    PREVIOUS_TOTAL, *LAG_COLUMNS
]
CATEGORICAL_FEATURES = ["State"]

# 4. BUILD MODEL PIPELINE
def make_model():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])
    category_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    prep = ColumnTransformer([
        ("numeric", numeric_pipe, NUMERIC_FEATURES),
        ("state", category_pipe, CATEGORICAL_FEATURES)
    ])
    return Pipeline([
        ("preprocessing", prep),
        ("regressor", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=4,
            loss="huber",
            random_state=42
        ))
    ])

# 5. CHRONOLOGICAL TEST: TRAIN THROUGH 2021, TEST 2022 ONWARD
train = data[data["Year"] <= 2021].copy()
test = data[data["Year"] >= 2022].copy()
if train.empty or test.empty:
    raise ValueError("The dataset needs training years through 2021 and test years from 2022 onward.")

evaluation_model = make_model()
evaluation_model.fit(train[FEATURES], train["Capacity Change"])
change_pred = evaluation_model.predict(test[FEATURES])
previous = test[PREVIOUS_TOTAL].to_numpy()
actual = test[TARGET].to_numpy()
prediction = np.maximum(previous + change_pred, 0)
baseline = previous.copy()

def metric_row(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "WAPE (%)": np.abs(y_true-y_pred).sum() / max(np.abs(y_true).sum(), 1e-9) * 100
    }

metrics = pd.DataFrame([
    metric_row("Persistence baseline", actual, baseline),
    metric_row("Gradient Boosting", actual, prediction)
])
print("\nMODEL EVALUATION — HELD-OUT YEARS")
display(metrics.round(3))
print("Note: Gradient Boosting does not necessarily beat the baseline on every metric.")

# 6. TRAIN FINAL MODEL ON ALL AVAILABLE HISTORICAL TRAINING ROWS
final_model = make_model()
final_model.fit(data[FEATURES], data["Capacity Change"])
joblib.dump({
    "model": final_model,
    "features": FEATURES,
    "capacity_columns": CAPACITY_COLUMNS,
    "target": TARGET
}, "renewable_energy_final_model.joblib")
print("\nFinal model trained and saved.")

# 7. SAMPLE INPUT — KARNATAKA, PREDICTING 2025 FROM 2024 VALUES
# Edit these values to try another state/region or a different forecast year.
# Population and GSDP here are example inputs copied from the dataset's
# Karnataka 2024 row; for a real forecast, enter estimates for the forecast year.
SAMPLE_INPUT = {
    "State": "Karnataka",
    "Forecast_Year": 2025,
    "Population": 68115063,
    "GSDP (crore INR)": 1221552.625,
    "Previous Year - Installed - Bio-Mass Power": 1907.72,
    "Previous Year - Installed - Small Hydro Power": 1280.73,
    "Previous Year - Installed - Solar Power": 17752.74,
    "Previous Year - Installed - Wind Power": 6019.61,
    "Previous Year - Installed - Waste to Energy": 19.42
}

def predict_renewable_capacity(user_input, trained_model=final_model):
    required_input = [
        "State", "Forecast_Year", "Population", "GSDP (crore INR)",
        *LAG_COLUMNS
    ]
    absent = [key for key in required_input if key not in user_input]
    if absent:
        raise ValueError(f"Please provide these input fields: {absent}")
    row = {
        "State": str(user_input["State"]),
        "Year": int(user_input["Forecast_Year"]),
        "Population": float(user_input["Population"]),
        "GSDP (crore INR)": float(user_input["GSDP (crore INR)"]),
        PREVIOUS_TOTAL: sum(float(user_input[c]) for c in LAG_COLUMNS)
    }
    for col in LAG_COLUMNS:
        row[col] = float(user_input[col])
    if any(row[c] < 0 for c in LAG_COLUMNS):
        raise ValueError("Previous-year capacity values cannot be negative.")
    if row["Population"] < 0 or row["GSDP (crore INR)"] < 0:
        raise ValueError("Population and GSDP cannot be negative.")
    X_new = pd.DataFrame([row], columns=FEATURES)
    estimated_change = float(trained_model.predict(X_new)[0])
    estimated_total = max(row[PREVIOUS_TOTAL] + estimated_change, 0.0)
    return {
        "State/Region": row["State"],
        "Forecast year": row["Year"],
        "Previous-year total capacity": row[PREVIOUS_TOTAL],
        "Estimated capacity change": estimated_change,
        "Predicted total renewable capacity": estimated_total
    }

sample_result = predict_renewable_capacity(SAMPLE_INPUT)
print("\nSAMPLE INPUT")
display(pd.DataFrame([SAMPLE_INPUT]).T.rename(columns={0: "Value"}))
print("\nSAMPLE PREDICTION")
display(pd.DataFrame([sample_result]).T.rename(columns={0: "Result"}))

# 8. OPTIONAL: CHANGE SAMPLE_INPUT ABOVE AND RUN THIS CELL AGAIN
# Example: change State, Forecast_Year, Population, GSDP and all five
# previous-year technology capacities, then run:
# display(pd.DataFrame([predict_renewable_capacity(SAMPLE_INPUT)]))

print("\nIMPORTANT:")
print("- Capacity unit is not specified in the supplied CSV; output stays in the dataset's original unit.")
print("- This is a one-year-ahead data-science estimate, not an official energy forecast.")
print("- Future Population and GSDP should be supplied as estimates for the forecast year.")
print("- The model is trained on the supplied historical dataset; unseen states are handled by the encoder.")
print("\nDownload the trained model file:")
files.download("renewable_energy_final_model.joblib")
